# Przygotowanie danych do analizy sentymentu tweetów

Notebook realizuje pierwszy etap projektu: wczytanie danych, preprocessing tekstu, automatyczne etykietowanie sentymentu metodą VADER oraz podstawową eksploracyjną analizę danych.

## Instalacja bibliotek

Ta komórka instaluje dodatkowe biblioteki potrzebne w notebooku. `vaderSentiment` służy do automatycznej analizy sentymentu, a `wordcloud` do przygotowania chmury słów.

In [ ]:
!pip install vaderSentiment wordcloud

## Import bibliotek

Ta komórka importuje biblioteki używane do pracy z danymi, czyszczenia tekstu, tworzenia wykresów, liczenia najczęstszych słów oraz analizy sentymentu metodą VADER.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

## Wczytanie danych

Ta komórka wczytuje plik `realdonaldtrump.csv` do ramki danych `df` i wyświetla pierwsze wiersze, aby sprawdzić, czy dane zostały poprawnie załadowane.

In [ ]:
df = pd.read_csv("realdonaldtrump.csv")

df.head()

## Informacje o danych

Ta komórka wyświetla podstawowe informacje o zbiorze danych: liczbę wierszy, nazwy kolumn, typy danych oraz liczbę wartości niepustych w każdej kolumnie.

In [ ]:
df.info()

## Nazwy kolumn

Ta komórka wyświetla listę kolumn dostępnych w zbiorze. Dzięki temu można potwierdzić, że kolumna z treścią tweeta nazywa się `content`.

In [ ]:
df.columns

## Wybór potrzebnych kolumn

Ta komórka ogranicza dane do kolumn przydatnych w dalszej analizie: identyfikatora tweeta, daty, treści, liczby retweetów, polubień, wzmianek i hashtagów.

In [ ]:
df = df[["id", "date", "content", "retweets", "favorites", "mentions", "hashtags"]].copy()

df.head()

## Kontrola braków i duplikatów

Ta komórka sprawdza liczbę wierszy, liczbę braków w kolumnie `content` oraz liczbę zduplikowanych treści tweetów.

In [ ]:
print("Liczba wierszy:", len(df))
print("Liczba braków w content:", df["content"].isna().sum())
print("Liczba duplikatów content:", df["content"].duplicated().sum())

## Usunięcie duplikatów

Ta komórka usuwa zduplikowane tweety na podstawie kolumny `content`. Dzięki temu ten sam tekst nie wpływa wielokrotnie na analizę.

In [ ]:
print("Liczba wierszy przed usunięciem duplikatów:", len(df))

df = df.drop_duplicates(subset=["content"]).copy()

print("Liczba wierszy po usunięciu duplikatów:", len(df))
print("Liczba duplikatów content po czyszczeniu:", df["content"].duplicated().sum())

## Definicja funkcji czyszczących tekst

Ta komórka definiuje dwie funkcje czyszczenia tekstu. `clean_tweet` tworzy mocniej oczyszczony tekst do późniejszego modelu TF-IDF, a `clean_for_vader` tworzy łagodniej oczyszczony tekst dla VADER-a, zachowując wielkie litery i interpunkcję, które mogą wpływać na wynik sentymentu.

In [ ]:
def clean_tweet(text):
    text = str(text)
    text = text.lower()

    # usunięcie linków
    text = re.sub(r"http\S+|www\S+", "", text)

    # usunięcie oznaczeń użytkowników, np. @username
    text = re.sub(r"@\w+", "", text)

    # usunięcie znaku hashtagu, ale zostawienie samego słowa
    text = re.sub(r"#", "", text)

    # podstawowe zamiany encji HTML
    text = re.sub(r"&amp;", "and", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)

    # zostawiamy tylko litery i spacje
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # usunięcie nadmiarowych spacji
    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean_for_vader(text):
    text = str(text)

    # usunięcie linków
    text = re.sub(r"http\S+|www\S+", "", text)

    # usunięcie oznaczeń użytkowników
    text = re.sub(r"@\w+", "", text)

    # zamiana encji HTML
    text = re.sub(r"&amp;", "and", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)

    # usunięcie nadmiarowych spacji
    text = re.sub(r"\s+", " ", text).strip()

    return text

## Utworzenie wersji tekstu po czyszczeniu

Ta komórka tworzy dwie nowe kolumny: `vader_text`, czyli tekst przygotowany do analizy VADER, oraz `clean_text`, czyli tekst mocniej oczyszczony do dalszego modelowania NLP.

In [ ]:
df["vader_text"] = df["content"].apply(clean_for_vader)
df["clean_text"] = df["content"].apply(clean_tweet)

df[["content", "vader_text", "clean_text"]].head(10)

## Usunięcie pustych tekstów po czyszczeniu

Ta komórka sprawdza, czy po czyszczeniu nie powstały puste teksty. Takie rekordy są usuwane, ponieważ nie nadają się do trenowania modelu ani do analizy sentymentu.

In [ ]:
print("Liczba wierszy przed usunięciem pustych clean_text:", len(df))

empty_clean_text = (df["clean_text"].str.len() == 0).sum()
print("Liczba pustych tekstów po czyszczeniu:", empty_clean_text)

df = df[df["clean_text"].str.len() > 0].copy()

print("Liczba wierszy po usunięciu pustych clean_text:", len(df))

## Dodanie długości tweetów

Ta komórka dodaje kolumny opisujące długość tekstu: długość oryginalnej treści, długość tekstu po czyszczeniu oraz liczbę słów. Te wartości są później wykorzystywane w analizie eksploracyjnej.

In [ ]:
df["original_length"] = df["content"].astype(str).apply(len)
df["clean_length"] = df["clean_text"].apply(len)
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

df[["content", "clean_text", "original_length", "clean_length", "word_count"]].head()

## Obliczenie wyniku sentymentu VADER

Ta komórka tworzy analizator VADER i oblicza dla każdego tweeta wartość `compound`, zapisaną w kolumnie `vader_score`. Wynik bliski `1` oznacza sentyment pozytywny, bliski `-1` sentyment negatywny, a okolice `0` tekst neutralny.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_vader_score(text):
    return analyzer.polarity_scores(text)["compound"]

df["vader_score"] = df["vader_text"].apply(get_vader_score)

df[["vader_text", "vader_score"]].head(10)

## Przypisanie etykiet sentymentu

Ta komórka zamienia liczbowy wynik `vader_score` na trzy etykiety: `positive`, `neutral` i `negative`. Zastosowano standardowe progi VADER: wartości od `0.05` oznaczają sentyment pozytywny, wartości do `-0.05` negatywny, a pozostałe neutralny.

In [ ]:
def get_sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["vader_score"].apply(get_sentiment_label)

df[["vader_text", "clean_text", "vader_score", "sentiment"]].head(20)

## Rozkład klas sentymentu

Ta komórka zlicza, ile tweetów przypisano do każdej klasy sentymentu oraz jaki procent całego zbioru stanowi każda klasa.

In [ ]:
sentiment_counts = df["sentiment"].value_counts()
sentiment_percentages = df["sentiment"].value_counts(normalize=True) * 100

print("Liczebność klas:")
print(sentiment_counts)

print("\nProcentowy udział klas:")
print(sentiment_percentages.round(2))

## Przykład ograniczenia VADER: słowo `miss`

Ta komórka sprawdza, jak VADER ocenia pojedyncze słowo `miss`. Jest to pomocne do pokazania ograniczeń metody, ponieważ `miss` może mieć negatywne znaczenie, ale w nazwach własnych typu `Miss Universe` nie oznacza negatywnego sentymentu.

In [ ]:
analyzer.polarity_scores("miss")

## Przykład ograniczenia VADER: wyrażenie `miss universe`

Ta komórka pokazuje, że VADER nadal ocenia wyrażenie `miss universe` jako lekko negatywne, ponieważ traktuje słowo `miss` słownikowo, a nie jako część nazwy własnej konkursu.

In [ ]:
analyzer.polarity_scores("miss universe")

## Wykres rozkładu sentymentu

Ta komórka tworzy wykres słupkowy pokazujący liczebność klas sentymentu i zapisuje go do pliku `sentiment_distribution.png`.

In [ ]:
plt.figure(figsize=(8, 5))

sentiment_counts.plot(kind="bar")

plt.xlabel("Klasa sentymentu")
plt.ylabel("Liczba tweetów")
plt.title("Rozkład klas sentymentu")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig("sentiment_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Histogram długości tweetów

Ta komórka tworzy histogram liczby słów w tweetach po czyszczeniu. Wykres pokazuje, jak długie są teksty używane później do modelowania, i zapisuje go do pliku `tweet_length_distribution.png`.

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df["word_count"], bins=30)

plt.xlabel("Liczba słów w tweecie")
plt.ylabel("Liczba tweetów")
plt.title("Rozkład długości tweetów po czyszczeniu")
plt.tight_layout()

plt.savefig("tweet_length_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Statystyki długości tweetów

Ta komórka wyświetla statystyki opisowe dla liczby słów w tweetach, między innymi średnią, odchylenie standardowe, medianę, minimum i maksimum.

In [ ]:
df["word_count"].describe()

## Podstawowa lista stop words

Ta komórka definiuje zbiór często występujących angielskich słów, które zwykle nie niosą istotnej informacji tematycznej, np. `the`, `and`, `to`.

In [ ]:
stop_words = {
    "the", "and", "to", "of", "a", "in", "for", "is", "on", "it", "that",
    "this", "with", "be", "are", "was", "at", "as", "by", "from", "or",
    "an", "have", "has", "not", "will", "we", "you", "i", "they", "he",
    "she", "his", "her", "our", "their", "my", "your", "me", "but", "so",
    "do", "does", "did", "can", "could", "would", "should", "there",
    "here", "about", "into", "than", "then", "just", "what", "when",
    "where", "who", "why", "how", "all", "out", "up", "down", "over",
    "under", "again", "more", "most", "some", "any", "very"
}

## Najczęstsze słowa przed dodatkowym filtrowaniem

Ta komórka zlicza najczęściej występujące słowa w oczyszczonych tweetach po usunięciu podstawowych stop words. Ten etap pozwala zauważyć, czy w wynikach pojawiają się techniczne artefakty, np. fragmenty linków.

In [ ]:
all_words = []

for text in df["clean_text"]:
    words = text.split()
    words = [
        word for word in words
        if word not in stop_words and len(word) > 2
    ]
    all_words.extend(words)

word_counts = Counter(all_words)
most_common_words = word_counts.most_common(20)

most_common_words

## Dodatkowe stop words specyficzne dla zbioru

Ta komórka rozszerza listę stop words o słowa techniczne i bardzo oczywiste dla tego datasetu, np. `realdonaldtrump`, `twitter`, `com`, `pic`. Usunięcie ich ułatwia analizę bardziej znaczących słów.

In [ ]:
custom_stop_words = {
    "realdonaldtrump",
    "donald",
    "trump",
    "twitter",
    "com",
    "pic",
    "http",
    "https",
    "www",
    "amp",
    "rt"
}

stop_words = stop_words.union(custom_stop_words)

## Najczęstsze słowa po dodatkowym filtrowaniu

Ta komórka ponownie zlicza najczęstsze słowa po rozszerzeniu listy stop words. Wynik jest bardziej informacyjny, bo usuwa część artefaktów technicznych i oczywistych nazw z danych.

In [ ]:
all_words = []

for text in df["clean_text"]:
    words = text.split()
    words = [
        word for word in words
        if word not in stop_words and len(word) > 2
    ]
    all_words.extend(words)

word_counts = Counter(all_words)
most_common_words = word_counts.most_common(20)

most_common_words

## Wykres najczęstszych słów

Ta komórka tworzy poziomy wykres słupkowy dla 20 najczęściej występujących słów po czyszczeniu i zapisuje go do pliku `most_common_words.png`.

In [ ]:
words = [item[0] for item in most_common_words]
counts = [item[1] for item in most_common_words]

plt.figure(figsize=(10, 6))

plt.barh(words, counts)

plt.xlabel("Liczba wystąpień")
plt.ylabel("Słowo")
plt.title("Najczęściej występujące słowa po czyszczeniu")
plt.gca().invert_yaxis()
plt.tight_layout()

plt.savefig("most_common_words.png", dpi=300, bbox_inches="tight")
plt.show()

## Chmura słów

Ta komórka tworzy chmurę słów na podstawie przefiltrowanej listy słów i zapisuje ją do pliku `wordcloud.png`. Większy rozmiar słowa oznacza częstsze występowanie w zbiorze.

In [ ]:
text_for_wordcloud = " ".join(all_words)

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    max_words=100
).generate(text_for_wordcloud)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Chmura najczęściej występujących słów")
plt.tight_layout()

plt.savefig("wordcloud.png", dpi=300, bbox_inches="tight")
plt.show()

## Przykładowe tweety z każdej klasy

Ta komórka wyświetla po kilka przykładów tweetów przypisanych do klas `positive`, `neutral` i `negative`. Ułatwia to jakościową ocenę działania automatycznego etykietowania VADER.

In [ ]:
for label in ["positive", "neutral", "negative"]:
    print("\n" + "=" * 40)
    print(label.upper())
    print("=" * 40)

    examples = df[df["sentiment"] == label][
        ["content", "vader_text", "clean_text", "vader_score", "sentiment"]
    ].head(5)

    display(examples)

## Zapis przykładowych tweetów

Ta komórka zapisuje po 10 przykładowych tweetów z każdej klasy sentymentu do pliku `sentiment_examples.csv`. Plik może być użyty przy pisaniu raportu.

In [ ]:
example_columns = ["content", "vader_text", "clean_text", "vader_score", "sentiment"]

example_tweets = pd.concat([
    df[df["sentiment"] == label][example_columns].head(10)
    for label in ["positive", "neutral", "negative"]
])

example_tweets.to_csv("sentiment_examples.csv", index=False)

example_tweets

## Zapis finalnego przetworzonego zbioru danych

Ta komórka zapisuje finalny plik `trump_tweets_preprocessed.csv`, który zawiera dane po czyszczeniu, wyniki VADER oraz etykiety sentymentu. Jest to główny plik przekazywany do kolejnego etapu projektu.

In [ ]:
output_columns = [
    "id",
    "date",
    "content",
    "vader_text",
    "clean_text",
    "original_length",
    "clean_length",
    "word_count",
    "retweets",
    "favorites",
    "mentions",
    "hashtags",
    "vader_score",
    "sentiment"
]

df[output_columns].to_csv("trump_tweets_preprocessed.csv", index=False)

print("Zapisano plik trump_tweets_preprocessed.csv")
print("Liczba zapisanych wierszy:", len(df))
print("Liczba zapisanych kolumn:", len(output_columns))

## Kontrola zapisanego pliku

Ta komórka ponownie wczytuje zapisany plik `trump_tweets_preprocessed.csv` i sprawdza jego wymiary oraz listę kolumn. To potwierdza, że plik został poprawnie utworzony i może być użyty przez kolejne osoby w projekcie.

In [ ]:
df_check = pd.read_csv("trump_tweets_preprocessed.csv")

print("Wymiary pliku:", df_check.shape)
print("\nKolumny:")
print(df_check.columns.tolist())

df_check.head()